In [ ]:
import pickle
import datetime
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import torch
from copy import deepcopy as dc
import os
from create_agents import create_flappy_agent, create_lunarlander_agent, create_robot_agent
import gymnasium as gym
import os
import re

## Overview
Pull fNIRS data file from dataset. Then align the neural data with the task data.

Add the labels, and then save it the file.

### Extract PID and Task Condition

In [ ]:
# PID
pid = 3

#Task and Condition
condition = "RW"

### Find Files for Participant Conditions

In [ ]:
pid = str("0" + str(pid) + "_")

def find_files(participant_id, condition):

    source_folder_1 = "/Users/[USER]/Desktop/fNIRS-2-RL/Experiment/ParticipantData/TaskData/raw"
    source_folder_2 = "/Users/[USER]/Desktop/fNIRS-2-RL/Experiment/ParticipantData/fNIRS/FilteredData"

    def find_matching_files_with_paths(folder, pid, condition):
        return [
            os.path.join(folder, file)
            for file in os.listdir(folder)
            if pid in file and condition in file]

    matching_files_folder1 = find_matching_files_with_paths(source_folder_1, participant_id, condition)
    matching_files_folder2 = find_matching_files_with_paths(source_folder_2, participant_id, condition)

    all_matching_files = {
        "Folder 1": matching_files_folder1,
        "Folder 2": matching_files_folder2
    }

    for folder, files in all_matching_files.items():
        print(f"\nMatching files in {folder}:")
        if files:
            for file in files:
                if file.endswith('.pickle'):
                    df_pickle = file
                if file.endswith('.csv'):
                    df_neural = file
                print(file)
        else:
            print("No matching files found.")

    if condition[0] == "L":
        policy_path = 'policies/LunarLanderPolicies'
    if condition[0] == "F":
        policy_path = 'policies/FlappyBirdPolicies'
    if condition[0] == "R":
        policy_path = 'policies/RobotPolicies'

    return df_pickle, df_neural, policy_path

In [ ]:
def policy_set():
    if condition[0] == "L":
        policy_set = {
            0: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy100_1"),
            1: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy100"),
            2: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy98"),
            3: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy98_1"),
            4: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy98_2"),
            5: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy98_3"),
            6: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy96"),
            7: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy96_1"),
            8: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy96_2"),
            9: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy96_3")
        }
    if condition[0] == "R":
        policy_set = {
            0: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace1.pth"),
            1: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace2.pth"),
            2: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace3.pth"),
            3: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace4.pth"),
            4: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace5.pth"),
            5: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace6.pth"),
            6: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace7.pth"),
            7: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace8.pth"),
            8: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace9.pth"),
            9: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace10.pth"),
        }
        
    if condition[0] == "F":
        policy_set = {
            0: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy1"),
            1: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy2"),
            2: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy3"),
            3: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy4"),
            4: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy5"),
            5: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy6"),
            6: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy7"),
            7: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy8"),
            8: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy9"),
            9: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy10"),

        }

    return policy_set

In [ ]:
# Function to extract PID and what condition it is from filename
def extract_pid_and_suffix(filename):
    pid_match = re.search(r'(\d{3})', filename)
    suffix_match = re.search(r'_(LP|LW|FP|FW|RP|RW)_', filename)
    pid = pid_match.group(1) if pid_match else None
    suffix = suffix_match.group(1) if suffix_match else None
    return pid, suffix

In [ ]:
demo_dfs, neural_dfs = [], []

# Load data files
df_pickle, df_neural, policy_path = find_files(pid, condition)
df_pickle = open(df_pickle, 'rb')
df_dict = dict(pickle.load(df_pickle))
df = pd.DataFrame(df_dict)
demo_dfs.append(df)

fnirs_df = pd.read_csv(df_neural)
neural_dfs.append(fnirs_df)

In [ ]:
#Neural features
all_neural_features = ['L_O_DSphi', 'L_D_DSphi', 'R_D_DSphi', 'R_O_DSphi', 'L_O_DSI', 'R_D_DSI', 'R_O_DSI', 'L_D_DSI']
phasic_neural_features = ['L_O_DSphi', 'L_D_DSphi', 'R_D_DSphi', 'R_O_DSphi']
intensity_neural_features = ['L_O_DSI', 'R_D_DSI', 'R_O_DSI', 'L_D_DSI']

#Labels
labels = ['continuous_optimal', 'binary_optimal', 'discrete_optimal']

### Classifier Functions:
#### Compute Discounted Rewards

In [ ]:
def compute_discounted_rewards(rewards, gamma):
    discounted_rewards = np.zeros_like(rewards, dtype=np.float64)
    cumulative_sum = 0.0
    
    for t in (range(len(rewards))):
        if rewards[t] == None:
            continue
        cumulative_sum = rewards[t] + gamma * cumulative_sum
        discounted_rewards[t] = cumulative_sum

    return discounted_rewards

### Align Datasets

In [ ]:
def change_neural_timestamps(neural_df, demo_df):
    neural_start_time = neural_df["time"][0]
    neural_timestamp = datetime.strptime(neural_start_time[0:25], "%Y-%m-%d %H:%M:%S.%f")

    print("Neural Start Time", demo_df["floatTimestamps"])
    if abs(demo_df["floatTimestamps"][1] - demo_df["floatTimestamps"][0]) > 15:
        demo_start_time = demo_df["floatTimestamps"][1]
    else:
        demo_start_time = demo_df["floatTimestamps"][0]
        
    diff = demo_start_time - neural_timestamp.timestamp()
    print("Demo Start Time", datetime.fromtimestamp(demo_start_time))

    neural_df[['dateTimestamps', 'floatTimestamps']] = neural_df['time'].apply(lambda x: pd.Series(add_time(x, diff)))
    neural_df["L_D_DSI"].size
    neural_df.drop(columns=['Unnamed: 0', 'time'], axis=0)

    return neural_df, demo_df

def add_time(timestamp_str, diff):
    timestamp_obj = datetime.strptime(timestamp_str[:-3], "%Y-%m-%d %H:%M:%S.%f")
    new_timestamp_obj = timestamp_obj + timedelta(seconds=diff)
    new_timestamp_str = new_timestamp_obj.strftime("%Y-%m-%d %H:%M:%S.%f") #+ timestamp_obj[-3:]
    new_float_timestamp = new_timestamp_obj.timestamp()

    return new_timestamp_str, new_float_timestamp

### Merge Datasets

In [ ]:
def merge_dfs(demo_df, neural_df, column_of_interest, time_window_seconds=0.1):
    demo_df['timestamps'] = pd.to_datetime(demo_df['floatTimestamps'], unit='s')
    neural_df['timestamps'] = pd.to_datetime(neural_df['floatTimestamps'], unit='s')

    print("Demo Data Length: ", len(demo_df['timestamps']))
    print("Neural Data Length: ", len(neural_df['timestamps']))
    print("Last Neural Timestamp: ", neural_df["timestamps"][len(neural_df['timestamps'])-1])

    merged_records = []

    for i, row in neural_df.iterrows():
        timestamp = row['timestamps']

        time_window = pd.Timedelta(seconds=time_window_seconds)
        demo_window = demo_df[
            (demo_df['timestamps'] >= timestamp - time_window) &
            (demo_df['timestamps'] <= timestamp + time_window)
        ]

        if not demo_window.empty:
            # Find the row with the maximum value in the specified column
            max_row = demo_window.loc[demo_window[column_of_interest].idxmax()]

            #Append the merged data
            merged_row = {**row.to_dict(), **max_row.to_dict()}
            merged_records.append(merged_row)

    # Convert the list of merged records to a DataFrame
    merged_df = pd.DataFrame(merged_records)

    return merged_df

### Create new Dictionary

In [ ]:
def create_time_dict(num, demo_dict, gamma):

    if condition[0]=="R":
        new_dict = {'floatTimestamps':[],
                    'dateTimestamps': [],
                    'episode': [],
                    'actions': [],
                    'rewards':[],
                    'discounted_rewards':[],
                    'chosen_actions': [],
                    'optimal_actions': [],
                    'desired_goal':[],
                    'states':[]}
    else:
        new_dict = {'floatTimestamps':[],
                    'dateTimestamps': [],
                    'episode': [],
                    'actions': [],
                    'rewards':[],
                    'discounted_rewards':[],
                    'chosen_actions': [],
                    'optimal_actions': [],
                    'states':[]}


    for i in range(num):
        try:
            print(demo_dict[i].keys())
        except:
            continue

        if condition[0]=="R":
                env = gym.make('FetchPickAndPlace-v2', max_episode_steps=650)
                seed = demo_dict[i]["seed"]
                state_dict, _ = env.reset(seed=seed) #reset
                desired_goal = state_dict["desired_goal"]
                print(desired_goal)

        r = compute_discounted_rewards(demo_dict[i]['rewards'], gamma)
        for j in range(len(demo_dict[i]['rewards'])):
            new_dict['floatTimestamps'].append(demo_dict[i]["timestamps"][j])
            new_dict['episode'].append(i)
            new_dict['dateTimestamps'].append(demo_dict[i]["timestamps_datetime"][j])
            new_dict['rewards'].append(demo_dict[i]["rewards"][j])

            try:
                new_dict['chosen_actions'].append(demo_dict[i]["chosen_actions"][j])
                new_dict['optimal_actions'].append(demo_dict[i]["optimal_actions"][j])
            except:
                new_dict['chosen_actions'].append(demo_dict[i]["chosen_action_prob"][j])
                new_dict['optimal_actions'].append(demo_dict[i]["optimal_action_prob"][j])

            try:
                new_dict['actions'].append(demo_dict[i]["actions"][j])
            except:
                new_dict['actions'].append(demo_dict[i]["chosen_actions"][j])

            new_dict['states'].append(demo_dict[i]["states"][j])
            new_dict['discounted_rewards'].append(r[j])

            if condition[0]=="R":
                new_dict["desired_goal"].append(desired_goal)
        
    return new_dict

### Util Functions

In [ ]:
def softmax(values):
    if values is None:
        return 0
 
    exp_values = np.exp(values, dtype=np.longdouble)
    exp_values_sum = np.sum(exp_values)
    vals = np.asarray(exp_values/exp_values_sum, dtype=np.float64)

    return vals

def from_tensor(x):
    try:
        x = x[0].detach().cpu().numpy()
        return softmax(x)
    except:
        return None

### Optimality Metric/Label Helper Functions

In [ ]:
def KL_regression_labels(P, Q):
    from scipy.special import rel_entr

    if P is None or Q is None:
        return 0

    return sum(rel_entr(P, Q))

def euclideanDist(vector1, vector2):
    if vector1 is None or vector2 is None:
        return 0
    
    return np.linalg.norm(vector1 - vector2)

def MSE(vector1, vector2):
    if vector1 is None or vector2 is None:
        return 0
    
    return np.mean((vector1 - vector2) ** 2)

def cosine_similarity(vector1, vector2):
    if vector1 is None or vector2 is None:
        return 0
    
    return np.dot(vector1, vector2) / (np.linalg.norm(vector1) * np.linalg.norm(vector2))

def discrete_labels():
    pass

def game_result(r, condition, conditional = None):
    if condition[0] == "L":
        if r > 75:
            return 0, 0
        elif r < -10:
            return 1, 2
        else:
            return 1, 1
        
    if condition[0] == "F":
        if r < 100:
            return 1, 2
        elif r < 200:
            return 1, 1
        else:
            return 0, 0
        
    if condition[0] == "R":
        if r > -1.0:
            return 0, 0
        elif conditional > 200:
            return 1, 2
        else:
            return 1, 1

def binary_optimal_labels(df, condition):
    from math import dist

    # Initialize new columns if they don't exist
    df["binary_optimal"] = 0
    df["discrete_optimal"] = 0

    for episode, group in df.groupby("episode"):
        conditional = group["rewards"].iloc[-1]
        reward = group["rewards"].iloc[-1]
        chosen_actions = group["chosen_actions"]
        optimal_actions = group["optimal_actions"]


        if condition[0] == "F":
            conditional = len(group)
            print(conditional)
        elif condition[0] == "R":
            reward = conditional
            conditional = 0
            for i, (o,a) in enumerate(zip(optimal_actions, chosen_actions)):
                if a is None or o is None:
                    continue
                distance = dist(a,o)

                conditional += distance
            print(conditional)

        group.head(1)


        binary, discrete = game_result(reward, condition, conditional)
        
        print(binary, discrete)

        max_index = group["continuous_optimal"].idxmax()

        if episode == 2:
            print(max_index, conditional, binary, discrete)

        df.loc[group.index[group.index >= max_index], "binary_optimal"] = binary
        df.loc[group.index[group.index >= max_index], "discrete_optimal"] = discrete

    return df


def CE_regression_labels(y_pred, y_true):
    if y_pred is None or y_true is None:
        return 0
    
    y_pred = softmax(y_pred)
    loss = 0
     
    for i in range(len(y_pred)):
        loss = loss + (-1 * y_true[i]*np.log(y_pred[i]))
 
    return loss

def robot_optimality_selection(policy_set, state, chosen_action, goal):
    error_values = []

    for policy_index in range(len(policy_set)):
        agent = policy_set[policy_index]

        optimal_action = agent.choose_action(state, goal, train_mode=False)

        error = euclideanDist(chosen_action, optimal_action)
      
        error_values.append(error)
        
    error_mean = np.mean(error_values, axis=0)

    return error_mean


def game_optimality_selection(policy_set, state, chosen_action_values, goal=None):
    prob_set = []
    all_values = []

    for policy_index in range(len(policy_set)):
        agent = policy_set[policy_index]

        _, optimal_action_values = agent.chooseAction(state, 0.0)
        optimal_action_values = softmax(from_tensor(optimal_action_values))
        all_values.append(optimal_action_values)

        if condition[1] == "W":
            error = KL_regression_labels(chosen_action_values, optimal_action_values)
        else:
            error = CE_regression_labels(chosen_action_values, optimal_action_values)

        prob_set.append(error)

    prob_set_mean = np.mean(np.asarray(prob_set))

    return prob_set_mean

def continuous_labels(policy_set, chosen_action_values, state, desired_goal=None):
    if condition[0] == "L" or condition[0] == "F":
        return game_optimality_selection(policy_set=policy_set, state=state, chosen_action_values=chosen_action_values, goal=None)
    else:
        return robot_optimality_selection(policy_set=policy_set, state=state, chosen_action=chosen_action_values, goal=desired_goal)

### Computing Labels for Dataset

In [ ]:
# Dictionary of metric functions
metrics = {
    'cross_entropy': CE_regression_labels,
    'euclidean_error': euclideanDist,
    'MSE_error': MSE,
    'cosine_similarity': cosine_similarity,
    'continuous_optimal': continuous_labels,
    'binary_optimal': binary_optimal_labels,
    'discrete_optimal': discrete_labels
}

def add_metric(df, metric_name):
    metric_func = metrics[metric_name]
    p_set = policy_set()
    # print(p_set[1])

    try:
        df[metric_name] = df.apply(
            lambda row: metric_func(p_set, row['chosen_actions'], row['states'], row['desired_goal']), axis=1
        )
    except:
        df[metric_name] = df.apply(
            lambda row: metric_func(p_set, row['chosen_actions'], row['states'], None), axis=1
        )

    print(df["continuous_optimal"].iloc(0))

def add_labels_to_df(df, binary_labels:bool = False, continuous_labels:bool= False, discrete_labels:bool = False):
    if continuous_labels:
        add_metric(df, 'continuous_optimal')
        
    if binary_labels:
        binary_optimal_labels()

    if discrete_labels:
        add_metric(df, 'discrete_optimal')
        
def drop_na(df):
    df = df.dropna()
    nans_per_column = df.isna().sum()
    print("NaNs Per Column After Dropping: ", nans_per_column)
    return df

### Main Function for Labeling Dataset

In [ ]:
for i, df in enumerate(demo_dfs):
    num = df["NumberOfDemos"]['seed']
    robot = False
   
    new_dict = create_time_dict(num, df, 0.99)
    # print("Goal", new_dict["desired_goal"][0])

    demo_df = pd.DataFrame(new_dict)
    demo_df.dropna(inplace=True)
    demo_df.reset_index(drop=True, inplace=True)


    try:
        demo_df["optimal_actions"] = demo_df['optimal_actions'].apply(lambda x: from_tensor(x) if torch.is_tensor(x) else x)
    except:
        demo_df["optimal_actions"] = demo_df['optimal_action_prob'].apply(lambda x: from_tensor(x) if torch.is_tensor(x) else x)

    add_labels_to_df(
        demo_df,
        binary_labels=False,
        discrete_labels=False,
        continuous_labels=True
    )

    dd = dc(demo_df)
    labeled_df = binary_optimal_labels(demo_df, condition=condition)
    print(labeled_df.size)

    neural_df, labeled_df = change_neural_timestamps(neural_df=neural_dfs[i], demo_df=labeled_df)
    merged_df = merge_dfs(demo_df=labeled_df, neural_df=neural_df, column_of_interest="continuous_optimal", time_window_seconds=0.4)
    merged_df = drop_na(merged_df.drop(index=0, axis=0))

In [ ]:
merged_df.to_csv("{}{}_LabeledData.csv".format(pid, condition))